In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from google.cloud import bigquery

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Helper Functions

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_helpers import (
    build_distribution_summary,
    build_outlier_summary,
    build_segment_summary,
    cast_numeric_fields,
    plot_metric_boxplot_views,
    plot_slices_of_segments_boxplot,
)

CHART_DIR = PROJECT_ROOT / "src" / "notebooks" / "charts"
TABLE_DIR = PROJECT_ROOT / "src" / "notebooks" / "tables"

# Statistic summary - Priced in April

In [3]:
PROJECT_ID = "gannett-datascience"
TABLE_ID = "gannett-datascience.test_results_zone.ss_test_result_v3-2"

NUMERIC_FIELDS = [
    "frequency",
    "breadth",
    "tenure",
    "tt_cost",
]
SEGMENT_FIELDS = [
    "src_risk_tier",
    "cohort",
    "Treatment",
    "contact_channel",
    "status",
    "contact_timing",
    "repeatedly_called",
]
IDS = [
    "billing_account",
    "id_subscrip",
    "email_date",
]

client = bigquery.Client(project=PROJECT_ID)

/Users/YKang1/PycharmProjects/stop_save_pChurn/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [4]:
query = f"""
SELECT
  {", ".join(NUMERIC_FIELDS + SEGMENT_FIELDS + IDS)}
FROM `{TABLE_ID}`
WHERE email_date < '2026-05-01'
"""
df = client.query(query).to_dataframe()
df = cast_numeric_fields(df, NUMERIC_FIELDS)
df["contact_channel_group"] = df["contact_channel"].replace(
    {
        "Online first": "Contacted both ways",
        "Called-In first": "Contacted both ways",
    }
)
risk_tier = df["src_risk_tier"].astype("string")
df["src_risk_tier"] = risk_tier.where(
    risk_tier.isna() | risk_tier.str.endswith(" risk"),
    risk_tier + " risk",
)
df_action = df[df["status"].ne("No Action yet")].copy()
df_no_action = df[df["status"].eq("No Action yet")].copy()
df_save = df[df["status"].eq("saved")].copy()
df_stop = df[df["status"].eq("stoped")].copy()

/Users/YKang1/PycharmProjects/stop_save_pChurn/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [5]:
population_summary = pd.DataFrame(
    [
        {
            "population": name,
            "accounts": frame["billing_account"].nunique(),
        }
        for name, frame in {
            "All": df,
            "Action": df_action,
            "No Action": df_no_action,
            "Saved": df_save,
            "Stopped": df_stop,
        }.items()
    ]
)
display(population_summary)

,population,accounts
0,All,9398
1,Action,1531
2,No Action,7867
3,Saved,589
4,Stopped,942


In [6]:
dfs = {
    "No Action": df_no_action,
    "Action": df_action,
    "Saved": df_save,
    "Stopped": df_stop,
}
comparison_dfs = {"Saved": df_save, "Stopped": df_stop}

distribution_summaries = pd.concat(
    {
        name: build_distribution_summary(frame, NUMERIC_FIELDS)
        for name, frame in dfs.items()
    },
    names=["dataset"],
).reset_index(level=0).reset_index(drop=True)
display(distribution_summaries)

,dataset,field,row_count,null_count,null_pct,zero_count,negative_count,min,max,mean,median,std,p01,p05,p10,p25,p50,p75,p90,p95,p99
0,No Action,frequency,7867,6,0.08,1632,0,0.00,91.00,38.46,29.00,35.37,0.00,0.00,0.00,2.00,29.00,77.00,90.00,91.00,91.00
1,No Action,breadth,7867,6,0.08,2466,0,0.00,21.00,4.18,3.00,4.24,0.00,0.00,0.00,0.00,3.00,7.00,10.00,12.00,16.00
2,No Action,tenure,7867,6,0.08,0,0,208.00,"27,867.00","1,382.95","1,354.00",886.36,324.00,337.00,504.00,997.00,"1,354.00","1,635.00","2,069.00","2,460.00","3,405.80"
3,No Action,tt_cost,7867,6,0.08,32,0,0.00,"9,279.71",571.25,502.83,439.16,107.89,112.89,166.27,374.25,502.83,704.96,949.38,"1,279.15","1,764.45"
4,Action,frequency,1531,3,0.20,183,0,0.00,91.00,45.90,46.00,34.53,0.00,0.00,0.00,9.00,46.00,82.00,90.00,91.00,91.00
5,Action,breadth,1531,3,0.20,292,0,0.00,20.00,5.04,4.00,4.19,0.00,0.00,0.00,1.00,4.00,8.00,11.00,13.00,15.73
6,Action,tenure,1531,3,0.20,0,0,323.00,"11,789.00","1,501.89","1,450.00",820.37,325.27,358.35,689.70,"1,144.00","1,450.00","1,728.25","2,332.00","2,639.65","3,790.92"
7,Action,tt_cost,1531,3,0.20,0,0,25.91,"3,925.74",504.38,462.87,336.87,108.23,119.76,169.16,289.29,462.87,621.14,826.45,"1,005.34","1,788.55"
8,Saved,frequency,589,2,0.34,28,0,0.00,91.00,56.41,67.00,32.41,0.00,1.00,5.60,26.00,67.00,88.00,91.00,91.00,91.00
9,Saved,breadth,589,2,0.34,64,0,0.00,18.00,5.69,5.00,4.06,0.00,0.00,0.00,2.00,5.00,9.00,11.00,13.00,16.00


In [7]:
segment_summaries = []
for segment in SEGMENT_FIELDS:
    for name, frame in dfs.items():
        summary = build_segment_summary(frame, segment, NUMERIC_FIELDS)
        summary = summary.rename(columns={segment: "segment_value"})
        summary.insert(0, "segment_field", segment)
        summary.insert(0, "dataset", name)
        segment_summaries.append(summary)

segment_summaries = pd.concat(segment_summaries, ignore_index=True)
display(segment_summaries)

,dataset,segment_field,segment_value,users,avg_frequency,median_frequency,p90_frequency,avg_breadth,median_breadth,p90_breadth,avg_tenure,median_tenure,p90_tenure,avg_tt_cost,median_tt_cost,p90_tt_cost
0,No Action,src_risk_tier,<NA>,3227,33.26,18.00,89.00,3.73,2.00,10.00,"1,351.97","1,362.00","1,820.00",514.24,474.19,816.96
1,No Action,src_risk_tier,1. Low risk,1901,71.93,84.00,91.00,6.70,7.00,12.00,"1,565.33","1,502.00","2,393.00",621.90,523.48,"1,151.23"
2,No Action,src_risk_tier,3. Medium risk,1267,10.08,3.00,33.00,1.70,0.00,6.00,"1,278.57","1,305.00","1,670.40",597.58,568.24,888.22
3,No Action,src_risk_tier,2. Med-Low risk,1065,40.28,41.00,71.60,5.03,5.00,10.00,"1,469.87","1,425.00","2,420.80",693.48,607.12,"1,282.92"
4,No Action,src_risk_tier,4. Med-High risk,375,6.44,3.00,18.00,1.41,0.00,4.60,879.26,840.00,"1,090.00",394.05,388.48,534.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,Action,repeatedly_called,0,1518,45.69,45.00,90.00,5.03,4.00,11.00,"1,500.95","1,450.00","2,328.00",505.15,463.20,826.59
69,Action,repeatedly_called,1,13,73.00,89.00,91.00,6.00,5.00,9.90,"1,619.92","1,432.00","2,777.30",407.48,418.10,701.71
70,Saved,repeatedly_called,0,576,56.06,66.00,91.00,5.68,5.00,11.00,"1,540.95","1,454.00","2,394.00",469.80,430.82,788.54
71,Saved,repeatedly_called,1,13,73.00,89.00,91.00,6.00,5.00,9.90,"1,619.92","1,432.00","2,777.30",407.48,418.10,701.71


# Business Boxplots

In [8]:
# No Action vs Had Action
df["action_group"] = np.where(
    df["status"].eq("No Action yet"),
    "No Action yet",
    "Had Action"
)
_ = plot_metric_boxplot_views(
    data=df,
    metrics=NUMERIC_FIELDS,
    group_col="action_group",
    group_order=["No Action yet", "Had Action"],
    chart_title="Metrics: No Action vs Had Action",
    chart_folder=str(CHART_DIR / "boxplots"),
    file_name="metrics_no_action_vs_had_action.png",
)

In [9]:
# Stoped vs Saved
# Repeated called vs Not
df_action["repeated_call_group"] = np.where(
    df_action["repeatedly_called"].fillna(0).astype(int).eq(1),
    "Repeatedly Called",
    "Called Once"
)
group_comparisons = [
    ("status", ["saved", "stoped"], "metrics_saved_vs_stoped.png"),
    (
        "repeated_call_group",
        ["Repeatedly Called", "Called Once"],
        "metrics_repeatedly_called_vs_called_once.png",
    ),
]

for col, order, file_name in group_comparisons:
    _ = plot_metric_boxplot_views(
        data=df_action,
        metrics=NUMERIC_FIELDS,
        group_col=col,
        group_order=order,
        chart_title=f"Metrics: {order[0]} vs {order[1]}",
        chart_folder=str(CHART_DIR / "boxplots"),
        file_name=file_name,
    )

In [10]:
# Stoped vs Saved outliers
outlier_summaries = pd.concat(
    {
        name: build_outlier_summary(frame, NUMERIC_FIELDS)
        for name, frame in comparison_dfs.items()
    },
    names=["dataset"],
).reset_index(level=0).reset_index(drop=True)
display(outlier_summaries)

,dataset,field,row_count,non_null_count,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,Saved,frequency,589,587,26.00,88.00,62.00,-67.00,181.00,0,0.00
1,Saved,breadth,589,587,2.00,9.00,7.00,-8.50,19.50,0,0.00
2,Saved,tenure,589,587,"1,148.00","1,777.50",629.50,203.75,"2,721.75",28,4.77
3,Saved,tt_cost,589,587,268.02,572.52,304.50,-188.72,"1,029.27",21,3.58
4,Stopped,frequency,942,941,4.00,76.00,72.00,-104.00,184.00,0,0.00
5,Stopped,breadth,942,941,1.00,8.00,7.00,-9.50,18.50,1,0.11
6,Stopped,tenure,942,941,"1,142.00","1,696.00",554.00,311.00,"2,527.00",53,5.63
7,Stopped,tt_cost,942,941,311.02,645.02,334.00,-189.98,"1,146.02",39,4.14


In [11]:
# Slices of Segments boxplots
slice_fields = [
    "contact_channel_group",
    "cohort",
    "src_risk_tier",
    "contact_timing",
]

segment_count_frames = []
for name, df_i in comparison_dfs.items():
    res = plot_slices_of_segments_boxplot(
        df_i,
        metrics=NUMERIC_FIELDS,
        slice_fields=slice_fields,
        chart_folder=str(CHART_DIR / "boxplots"),
        full_file_name=f"{name}_segment_slices_full.png",
        clipped_file_name=f"{name}_segment_slices_clipped.png",
        standardized_clipped_file_name=f"{name}_segment_slices_standardized_clipped.png",
    )
    population_counts = res.copy()
    population_counts.insert(0, "population", name)
    segment_count_frames.append(population_counts)

segment_combo_counts = (
    pd.concat(segment_count_frames, ignore_index=True)
    .sort_values(["population", *slice_fields, "Treatment"])
    .reset_index(drop=True)
)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
segment_counts_path = TABLE_DIR / "saved_stopped_segment_combo_counts.csv"
segment_combo_counts.to_csv(segment_counts_path, index=False)
segment_counts_path

PosixPath('/Users/YKang1/PycharmProjects/stop_save_pChurn/src/notebooks/tables/saved_stopped_segment_combo_counts.csv')

# Business profiles and retention hypotheses

These are directional archetypes, not mutually exclusive customer clusters. They combine the raw distribution summaries with observed Saved/Stopped shares among the 1,531 users who took an action.

## Profile 1: Engaged, agent-reachable subscriber — most characteristic of Saved

- The typical Saved account is materially more active: median frequency is 67 versus 33 for Stopped, while median breadth is 5 versus 4. Median tenure is nearly the same (1,454 versus 1,421), so recent engagement distinguishes the outcomes much more than customer age.
- Saved accounts also have a lower median `tt_cost` (428 versus 484). If `tt_cost` represents price or customer cost burden, this is consistent with stronger perceived value; its business definition must be confirmed before using that interpretation.
- Saved users are highly concentrated in the Called-In Cancel Flow: 486 of 589 Saved users called in. Among all action users, the observed Saved share is 50.8% for Called-In versus 17.3% for Online.
- Called-In users contacting on/after pricing are somewhat more saveable than those contacting before pricing (54.2% versus 48.5% observed Saved share). In the largest high-save slices, the share reaches 60.9% for two-offer on/after-pricing callers and 56.9% for low-risk three-offer on/after-pricing callers.
- **Business hypothesis:** current engagement signals willingness to stay, while live-agent contact creates an opportunity to resolve the immediate objection. Prioritize fast agent routing and personalized value reminders for highly engaged subscribers entering cancellation.

## Profile 2: Digital self-service exiter — most characteristic of Stopped

- Online Cancel Flow contains 458 of 942 Stopped users but only 96 of 589 Saved users. Across segment combinations with at least 20 action users, online observed Saved shares range from 11.3% to 25.7%, compared with 32.7% to 60.9% for called-in slices.
- The overall Stopped user is less active but not meaningfully newer: frequency is roughly half the Saved median, while median tenure differs by only 33 days on a roughly four-year base. This looks more like a long-standing subscriber whose product use has weakened than a newly acquired user failing to form a habit.
- Stopped users also carry somewhat higher `tt_cost`, suggesting a possible value-for-money objection if that field is cost-like. Lower engagement combined with higher cost is the clearest behavioral description of the Stopped population.
- **Business hypothesis:** the online path is where low-engagement, low-perceived-value customers can leave with little recovery opportunity. Test contextual value messaging, a targeted offer, or optional agent handoff inside the online flow; do not assume that forcing every user to call will reproduce the called-in result.

## Profile 3: Higher-risk, low-engagement account — an upstream prevention target

- Higher source-risk slices generally have lower frequency, breadth, tenure, and `tt_cost`. Observed Saved share declines from 43.8% at low risk to 34.7% at med-low and 27.1% at medium risk. The med-high estimate is 33.8% but is based on only 74 action users; the single high-risk user is not interpretable.
- This profile is visible in both outcomes, which suggests risk tier is identifying a broader low-engagement state rather than a unique treatment response. By the time these users enter a cancel flow, price presentation alone may be too late to rebuild value.
- **Business hypothesis:** treat falling engagement plus elevated risk as an earlier intervention trigger. Habit-building, content discovery, or service outreach should be tested before the pricing/cancellation moment.

## Profile 4: Highly engaged negotiated-price saver — small, post-outcome niche

- All 13 accounts flagged `repeatedly_called` are Saved and have median frequency 89, breadth 5, tenure 1,432, and `tt_cost` 418. They resemble highly engaged customers willing to negotiate rather than disengaged churners.
- The SQL derives this flag from a post-contact payment below the offered rate, not directly from a second-call event. It therefore contains outcome leakage and cannot be used as a prospective targeting feature.
- **Business hypothesis:** audit these exception cases to understand which concessions closed the save and whether the resulting price was intentional and economically acceptable.

## Cross-profile business implications

- **Engagement and channel are the useful profiling axes; tenure is not.** A retention strategy based only on loyalty/tenure would miss the largest observed differences.
- **Offer cohort alone does not define a customer type:** two-offer and three-offer cohorts have nearly identical observed Saved shares (38.7% and 38.3%).
- **Treatment needs a separate effectiveness audit:** observed Saved shares are 58.0% for Control, 29.0% for Midpoint, and 20.6% for Tiered, and Control remains higher within both cohorts and channels. This counterintuitive pattern is more important than small within-outcome boxplot shifts, but it should be validated against randomization, eligibility, outcome maturity, and risk balance before drawing an experimental conclusion.

## Interpretation boundaries

- Observed Saved shares are descriptive ratios among users with a classified action outcome; channel and timing are self-selected and should not be interpreted causally.
- User counts are unique `billing_account` values. Segment treatment groups below `min_n=5` are omitted from boxplots, and profile slices below 20 total action users are excluded from the profile table.
- Saved and Stopped standardized charts use separate population medians and IQRs. Compare standardized patterns within one population; use raw summaries or clipped raw views to compare magnitudes across outcomes.
- Full-value charts retain the long tails, clipped charts show typical business magnitudes, and standardized clipped charts support relative cross-metric comparison.

# WIP

In [18]:
# distribution_summary.to_csv("distribution_summary.csv")
# quality_df.to_csv("data_quality_summary.csv", index=False)
# outlier_df.to_csv("outlier_summary.csv", index=False)

# print("Saved summary files.")

In [19]:
# percentile_query = f"""
# SELECT
#   COUNT(*) AS row_count,

#   APPROX_QUANTILES(tenure, 100)[OFFSET(1)] AS tenure_p01,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(25)] AS tenure_p25,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(50)] AS tenure_p50,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(75)] AS tenure_p75,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(90)] AS tenure_p90,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(95)] AS tenure_p95,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(99)] AS tenure_p99,

#   APPROX_QUANTILES(total_cost, 100)[OFFSET(1)] AS total_cost_p01,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(25)] AS total_cost_p25,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(50)] AS total_cost_p50,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(75)] AS total_cost_p75,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(90)] AS total_cost_p90,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(95)] AS total_cost_p95,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(99)] AS total_cost_p99,

#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(1)] AS pages_p01,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(25)] AS pages_p25,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(50)] AS pages_p50,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(75)] AS pages_p75,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(90)] AS pages_p90,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(95)] AS pages_p95,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(99)] AS pages_p99

# FROM `{TABLE_ID}`
# """

# percentiles_bq = client.query(percentile_query).to_dataframe()
# percentiles_bq